# Project Workspace

The models, vector database, and supporting files used in this notebook are stored in Google Drive. Mounting the drive allows the notebook to access these saved artifacts without recreating them, making the pipeline reusable across sessions.

In [1]:
# Mount Google Drive

from google.colab import drive

drive.mount("/content/drive")

Mounted at /content/drive


Now that the project files are accessible, the required libraries are imported. Most of these have already been used in the previous notebooks, while a few new ones are needed for loading the language model and generating responses.

In [2]:
# Standard libraries
import os
import re
import json
import pickle
import warnings

from pathlib import Path

# Data handling
import numpy as np
import pandas as pd

# PyTorch
import torch
import torch.nn as nn

# Sentence Transformers
from sentence_transformers import SentenceTransformer

# Vector Search
!pip install faiss-gpu
!pip install faiss-cpu
import faiss

# Hugging Face
from transformers import (
    AutoTokenizer,
    AutoModelForSeq2SeqLM,
    pipeline
)

warnings.filterwarnings("ignore")

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 134.9/134.9 MB 7.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 581.2/581.2 MB 3.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.5/3.5 MB 2.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 68.3/68.3 MB 12.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 89.6/89.6 MB 10.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 18.5/18.5 MB 75.0 MB/s eta 0:00:00


The notebook loads all previously generated files instead of creating them again. Keeping the paths in one place makes the code easier to maintain and simplifies the Streamlit integration later.

In [3]:
# Root directory of the project
PROJECT_DIR = Path("/content/drive/MyDrive/AI_Loan_Advisory_Chatbot")

# Important folders
PROCESSED_DIR = PROJECT_DIR / "processed"
EMBEDDING_DIR = PROCESSED_DIR / "embeddings"
VECTOR_DB_DIR = PROJECT_DIR / "vector_db"
MODEL_DIR = PROJECT_DIR / "models"

print("Project directory:", PROJECT_DIR)

Project directory: /content/drive/MyDrive/AI_Loan_Advisory_Chatbot


All saved models and supporting files are referenced here. These files were generated in the earlier notebooks and will be loaded throughout the RAG pipeline.

In [4]:
# Embedding files
EMBEDDING_FILE = EMBEDDING_DIR / "embeddings.pkl"
METADATA_FILE = EMBEDDING_DIR / "metadata.pkl"

# FAISS index
FAISS_INDEX_FILE = VECTOR_DB_DIR / "faiss_index.bin"

# Intent classifier
GRU_MODEL_FILE = MODEL_DIR / "gru_intent_model.pth"
VOCAB_FILE = MODEL_DIR / "vocabulary.pkl"
TOKENIZER_FILE = MODEL_DIR / "tokenizer.pkl"
LABEL_ENCODER_FILE = MODEL_DIR / "label_encoder.pkl"

Before loading anything, it's a good idea to verify that all required files are available. This avoids unexpected errors later if a model or index is missing.

In [5]:
required_files = {
    "Embeddings": EMBEDDING_FILE,
    "Metadata": METADATA_FILE,
    "FAISS Index": FAISS_INDEX_FILE,
    "GRU Model": GRU_MODEL_FILE,
    "Vocabulary": VOCAB_FILE,
    "Tokenizer": TOKENIZER_FILE,
    "Label Encoder": LABEL_ENCODER_FILE
}

print("Checking project files...\n")

missing_files = []

for name, path in required_files.items():
    if path.exists():
        print(f"✓ {name}")
    else:
        print(f"✗ {name} -> {path}")
        missing_files.append(name)

if not missing_files:
    print("\nAll required files are available.")
else:
    print("\nMissing files:", ", ".join(missing_files))

Checking project files...

✓ Embeddings
✓ Metadata
✓ FAISS Index
✓ GRU Model
✓ Vocabulary
✓ Tokenizer
✓ Label Encoder

All required files are available.


The notebook automatically detects whether a GPU is available. If CUDA is enabled in Colab, PyTorch will use it; otherwise, the pipeline runs on the CPU without requiring any code changes.

In [6]:
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")

print(f"Running on: {DEVICE}")

if DEVICE.type == "cuda":
    print("GPU :", torch.cuda.get_device_name(0))

Running on: cpu


User queries are converted into dense vector embeddings before performing similarity search. The same embedding model used while building the FAISS index is loaded here to keep the vector representations consistent.

In [7]:
EMBEDDING_MODEL_NAME = "sentence-transformers/all-MiniLM-L6-v2"

embedding_model = SentenceTransformer(
    EMBEDDING_MODEL_NAME,
    device=DEVICE.type
)

print("Embedding model loaded successfully.")

modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md:   0%|          | 0.00/10.5k [00:00<?, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/612 [00:00<?, ?B/s]

...

vocab.txt:   0%|          | 0.00/232k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/466k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

Embedding model loaded successfully.


The FAISS index stores vector representations of all document chunks. Instead of searching through every chunk manually, the query embedding is compared against this index to quickly retrieve the most relevant information.

In [8]:
# Load FAISS index

faiss_index = faiss.read_index(str(FAISS_INDEX_FILE))

print("FAISS index loaded successfully.")
print(f"Total vectors : {faiss_index.ntotal}")

FAISS index loaded successfully.
Total vectors : 241


The embeddings and metadata generated earlier are loaded into memory. While FAISS performs similarity search, the metadata helps identify the bank, loan type, source document and other details for each retrieved chunk.

In [9]:
# Load embeddings

with open(EMBEDDING_FILE, "rb") as f:
    document_embeddings = pickle.load(f)

# Load metadata

with open(METADATA_FILE, "rb") as f:
    document_metadata = pickle.load(f)

print(f"Embeddings loaded : {len(document_embeddings)}")
print(f"Metadata records  : {len(document_metadata)}")

Embeddings loaded : 241
Metadata records  : 241


Before moving further, a quick consistency check is performed. The number of vectors, embeddings and metadata entries should match to ensure that every retrieved result points to the correct document chunk.

In [10]:
print("=" * 50)
print("Resource Summary")
print("=" * 50)

print(f"FAISS vectors      : {faiss_index.ntotal}")
print(f"Embeddings         : {len(document_embeddings)}")
print(f"Metadata entries   : {len(document_metadata)}")

if (
    faiss_index.ntotal
    == len(document_embeddings)
    == len(document_metadata)
):
    print("\nAll resources are consistent.")
else:
    print("\nWarning: Resource count mismatch detected.")

Resource Summary
FAISS vectors      : 241
Embeddings         : 241
Metadata entries   : 241

All resources are consistent.


Two metadata files are available in the project. Before using one for retrieval, a quick inspection helps verify their structure and determine which file should be paired with the FAISS index.

In [11]:
# Metadata inside embeddings folder

with open(METADATA_FILE, "rb") as f:
    embedding_metadata = pickle.load(f)

# Metadata inside vector_db folder

FAISS_METADATA_FILE = VECTOR_DB_DIR / "faiss_metadata.pkl"

if FAISS_METADATA_FILE.exists():
    with open(FAISS_METADATA_FILE, "rb") as f:
        faiss_metadata = pickle.load(f)

    print("FAISS metadata loaded successfully.")
else:
    faiss_metadata = None
    print("faiss_metadata.pkl not found.")

FAISS metadata loaded successfully.


The first record from each metadata file is displayed for comparison. If both contain the same information, either can be used. Otherwise, the metadata created alongside the FAISS index is usually the safer choice.

In [12]:
print("=" * 60)
print("Embeddings Metadata")
print("=" * 60)

print(type(embedding_metadata))
print("Total records:", len(embedding_metadata))

print("\nFirst record:\n")
print(embedding_metadata[0])

if faiss_metadata is not None:
    print("\n" + "=" * 60)
    print("FAISS Metadata")
    print("=" * 60)

    print(type(faiss_metadata))
    print("Total records:", len(faiss_metadata))

    print("\nFirst record:\n")
    print(faiss_metadata[0])

Embeddings Metadata
<class 'list'>
Total records: 241

First record:

{'chunk_id': 1, 'bank': 'HDFC', 'loan_type': 'Car Loan', 'document_type': 'MITC', 'section': 'Clause 2', 'source_file': 'HDFC_Car_Loan_MITC.pdf', 'page_number': 'N/A', 'chunk_text': "1. I agree to abide by the Bank's Terms and Conditions and rules in force and the changes thereto in\nTerms and Conditions from time to time relating to my account as communicated and made available\non the Bank's website.", 'word_count': 40}

FAISS Metadata
<class 'list'>
Total records: 241

First record:

{'chunk_id': 1, 'bank': 'HDFC', 'loan_type': 'Car Loan', 'document_type': 'MITC', 'section': 'Clause 2', 'source_file': 'HDFC_Car_Loan_MITC.pdf', 'page_number': 'N/A', 'chunk_text': "1. I agree to abide by the Bank's Terms and Conditions and rules in force and the changes thereto in\nTerms and Conditions from time to time relating to my account as communicated and made available\non the Bank's website.", 'word_count': 40}


Once the metadata structure is verified, a single reference is used throughout the notebook. This keeps the retrieval code clean and avoids changing multiple variables later.

In [13]:
# Use FAISS metadata if available

if faiss_metadata is not None:
    retrieval_metadata = faiss_metadata
    print("Using metadata from vector_db.")
else:
    retrieval_metadata = embedding_metadata
    print("Using metadata from embeddings folder.")

Using metadata from vector_db.


In [14]:
embedding_metadata[0]

{'chunk_id': 1,
 'bank': 'HDFC',
 'loan_type': 'Car Loan',
 'document_type': 'MITC',
 'section': 'Clause 2',
 'source_file': 'HDFC_Car_Loan_MITC.pdf',
 'page_number': 'N/A',
 'chunk_text': "1. I agree to abide by the Bank's Terms and Conditions and rules in force and the changes thereto in\nTerms and Conditions from time to time relating to my account as communicated and made available\non the Bank's website.",
 'word_count': 40}

In [15]:
faiss_metadata[0]

{'chunk_id': 1,
 'bank': 'HDFC',
 'loan_type': 'Car Loan',
 'document_type': 'MITC',
 'section': 'Clause 2',
 'source_file': 'HDFC_Car_Loan_MITC.pdf',
 'page_number': 'N/A',
 'chunk_text': "1. I agree to abide by the Bank's Terms and Conditions and rules in force and the changes thereto in\nTerms and Conditions from time to time relating to my account as communicated and made available\non the Bank's website.",
 'word_count': 40}

The intent classifier trained in the previous notebook is reused here without any retraining. The same network architecture is recreated before loading the saved model weights.

In [16]:
class GRUIntentClassifier(nn.Module):

    def __init__(self, vocab_size, embedding_dim,
                 hidden_dim, output_dim,
                 num_layers=1, dropout=0.2):

        super().__init__()

        self.embedding = nn.Embedding(
            vocab_size,
            embedding_dim,
            padding_idx=0
        )

        self.gru = nn.GRU(
            embedding_dim,
            hidden_dim,
            batch_first=True,
            num_layers=num_layers,
            dropout=dropout if num_layers > 1 else 0
        )

        self.dropout = nn.Dropout(dropout)

        self.fc = nn.Linear(hidden_dim, output_dim)

    def forward(self, x):

        x = self.embedding(x)

        _, hidden = self.gru(x)

        output = self.dropout(hidden[-1])

        return self.fc(output)

The same vocabulary and label encoder used during training are loaded to ensure that incoming queries are processed exactly as they were during model development.

In [17]:
# Load vocabulary
with open(VOCAB_FILE, "rb") as f:
    vocabulary = pickle.load(f)

# Load preprocessing configuration
with open(TOKENIZER_FILE, "rb") as f:
    preprocessing_config = pickle.load(f)

# Load label encoder
with open(LABEL_ENCODER_FILE, "rb") as f:
    label_encoder = pickle.load(f)

MAX_SEQUENCE_LENGTH = preprocessing_config["max_sequence_length"]
MIN_WORD_FREQUENCY = preprocessing_config["min_word_frequency"]

print(f"Vocabulary Size      : {len(vocabulary)}")
print(f"Number of Intents    : {len(label_encoder.classes_)}")
print(f"Maximum Sequence Len : {MAX_SEQUENCE_LENGTH}")

Vocabulary Size      : 229
Number of Intents    : 19
Maximum Sequence Len : 20


The classifier is restored from the saved checkpoint. Along with the model weights, the checkpoint also stores the training configuration, allowing the same architecture to be recreated automatically.

In [18]:
# Load model checkpoint

checkpoint = torch.load(
    GRU_MODEL_FILE,
    map_location=DEVICE
)

intent_model = GRUIntentClassifier(
    vocab_size=checkpoint["vocab_size"],
    embedding_dim=checkpoint["embedding_dim"],
    hidden_dim=checkpoint["hidden_dim"],
    output_dim=checkpoint["num_classes"],
    num_layers=checkpoint["num_layers"],
    dropout=checkpoint["dropout"]
).to(DEVICE)

intent_model.load_state_dict(checkpoint["model_state_dict"])
intent_model.eval()

print("Intent classifier loaded successfully.")

Intent classifier loaded successfully.


Before sending a query to the intent classifier, it is cleaned using the same preprocessing steps followed during training. This keeps inference consistent with the data the model has already seen.

In [19]:
def preprocess_text(text):
    """Clean and normalize user query."""

    text = text.lower()
    text = re.sub(r"[^a-z0-9\s]", " ", text)
    text = re.sub(r"\s+", " ", text).strip()

    return text

The cleaned query is converted into numerical token IDs using the saved vocabulary. Unknown words are mapped to the `<UNK>` token, and the sequence is padded to the same length used during model training.

In [20]:
def text_to_sequence(text):
    """Convert text into a padded sequence."""

    text = preprocess_text(text)

    tokens = text.split()

    unk_index = vocabulary.get("<UNK>", 1)
    pad_index = vocabulary.get("<PAD>", 0)

    sequence = [
        vocabulary.get(token, unk_index)
        for token in tokens
    ]

    sequence = sequence[:MAX_SEQUENCE_LENGTH]

    if len(sequence) < MAX_SEQUENCE_LENGTH:
        sequence.extend(
            [pad_index] * (MAX_SEQUENCE_LENGTH - len(sequence))
        )

    return torch.tensor(sequence, dtype=torch.long)

The trained GRU model predicts the most likely intent for a given query. Along with the predicted label, a confidence score is also returned so that uncertain predictions can be handled safely.

In [21]:
def predict_intent(query):
    """Predict intent for a user query."""

    sequence = text_to_sequence(query)

    sequence = sequence.unsqueeze(0).to(DEVICE)

    with torch.no_grad():
        output = intent_model(sequence)

        probabilities = torch.softmax(output, dim=1)

        confidence, prediction = torch.max(
            probabilities,
            dim=1
        )

    intent = label_encoder.inverse_transform(
        [prediction.item()]
    )[0]

    return {
        "intent": intent,
        "confidence": float(confidence.item())
    }

All loaded models and supporting files are grouped into a single object. This keeps the remaining functions independent of global variables and makes the pipeline easier to reuse in the Streamlit application.

In [22]:
def load_models():
    """Return all loaded resources."""

    return {
        "embedding_model": embedding_model,
        "faiss_index": faiss_index,
        "metadata": retrieval_metadata,
        "intent_model": intent_model,
        "vocabulary": vocabulary,
        "label_encoder": label_encoder,
        "max_sequence_length": MAX_SEQUENCE_LENGTH,
        "device": DEVICE
    }

The resource loader is executed once after all models have been restored. Every component required by the RAG pipeline is now available through a single dictionary.

In [23]:
resources = load_models()

print("Resources loaded successfully.\n")

print(resources.keys())

Resources loaded successfully.

dict_keys(['embedding_model', 'faiss_index', 'metadata', 'intent_model', 'vocabulary', 'label_encoder', 'max_sequence_length', 'device'])


A small summary is displayed to confirm that the major components have been loaded correctly before moving on to semantic retrieval.

In [24]:
print("=" * 55)
print("RAG Pipeline Resources")
print("=" * 55)

print(f"Embedding Model : {type(resources['embedding_model']).__name__}")
print(f"FAISS Vectors   : {resources['faiss_index'].ntotal}")
print(f"Metadata Chunks : {len(resources['metadata'])}")
print(f"Intent Classes  : {len(resources['label_encoder'].classes_)}")
print(f"Device          : {resources['device']}")

RAG Pipeline Resources
Embedding Model : SentenceTransformer
FAISS Vectors   : 241
Metadata Chunks : 241
Intent Classes  : 19
Device          : cpu


Every user query is converted into a dense vector before searching the document collection. Using the same embedding model for both documents and queries keeps the similarity search consistent.

In [25]:
def generate_query_embedding(query):
    """Generate embedding for a user query."""

    query = preprocess_text(query)

    embedding = embedding_model.encode(
        query,
        convert_to_numpy=True,
        normalize_embeddings=True
    )

    return embedding.astype(np.float32)

The query embedding is compared with all stored document embeddings using the FAISS index. The search returns the most similar document chunks along with their similarity scores.

In [26]:
def semantic_search(query_embedding, top_k=5):
    """Search the FAISS index."""

    scores, indices = faiss_index.search(
        np.array([query_embedding]),
        top_k
    )

    return scores[0], indices[0]

The retrieved indices are mapped back to their corresponding metadata records. Along with the document information, the similarity score is also preserved for later use during response generation.

In [27]:
def retrieve_documents(query, top_k=5):
    """Retrieve the most relevant document chunks."""

    query_embedding = generate_query_embedding(query)

    scores, indices = semantic_search(
        query_embedding,
        top_k=top_k
    )

    retrieved_chunks = []

    for score, idx in zip(scores, indices):

        if idx == -1:
            continue

        chunk = retrieval_metadata[idx].copy()
        chunk["similarity_score"] = float(score)

        retrieved_chunks.append(chunk)

    return retrieved_chunks

Some queries clearly refer to a specific loan category, while others are more general. When appropriate, the predicted intent is used to keep only the relevant loan documents without affecting broader queries.

In [28]:
LOAN_INTENT_MAPPING = {
    "home_loan": "Home Loan",
    "personal_loan": "Personal Loan",
    "education_loan": "Education Loan",
    "gold_loan": "Gold Loan",
    "vehicle_loan": "Vehicle Loan",
    "loan_against_property": "Loan Against Property",
    "car_loan": "Car Loan"
}


def filter_by_intent(retrieved_chunks, intent):
    """Filter retrieved chunks using the predicted intent."""

    loan_type = LOAN_INTENT_MAPPING.get(intent)

    if loan_type is None:
        return retrieved_chunks

    filtered_chunks = [
        chunk
        for chunk in retrieved_chunks
        if chunk["loan_type"] == loan_type
    ]

    return filtered_chunks if filtered_chunks else retrieved_chunks

The retrieval pipeline now combines semantic search with intent-guided filtering. FAISS first retrieves the most relevant chunks, after which the predicted intent is used to refine the final context when applicable.

In [29]:
def retrieve_documents(query, intent=None, top_k=15):
    """Retrieve relevant document chunks."""

    query_embedding = generate_query_embedding(query)

    scores, indices = semantic_search(
        query_embedding,
        top_k=top_k
    )

    retrieved_chunks = []

    for score, idx in zip(scores, indices):

        if idx == -1:
            continue

        chunk = retrieval_metadata[idx].copy()
        chunk["similarity_score"] = float(score)

        retrieved_chunks.append(chunk)

    if intent is not None:
        retrieved_chunks = filter_by_intent(
            retrieved_chunks,
            intent
        )

    return retrieved_chunks

A quick test is performed to verify that the predicted intent and the retrieved documents are working together before integrating the language model.

In [30]:
sample_query = "What is the interest rate for SBI Home Loan?"

intent_result = predict_intent(sample_query)

documents = retrieve_documents(
    sample_query,
    intent=intent_result["intent"],
    top_k=5
)

print(f"Query      : {sample_query}")
print(f"Intent     : {intent_result['intent']}")
print(f"Confidence : {intent_result['confidence']:.2f}")

print("\nRetrieved Documents:\n")

for i, doc in enumerate(documents, start=1):
    print(f"{i}. {doc['source_file']}")
    print(f"   Bank       : {doc['bank']}")
    print(f"   Loan Type  : {doc['loan_type']}")
    print(f"   Similarity : {doc['similarity_score']:.4f}\n")

Query      : What is the interest rate for SBI Home Loan?
Intent     : interest_rate
Confidence : 0.99

Retrieved Documents:

1. SBI_PERSONAL_LOAN_MITC.pdf
   Bank       : SBI
   Loan Type  : Personal Loan
   Similarity : 0.6940

2. SBI_Home_Loan_Process_Guide.pdf
   Bank       : SBI
   Loan Type  : Home Loan
   Similarity : 0.6853

3. SBI_Home_Loan_Process_Guide.pdf
   Bank       : SBI
   Loan Type  : Home Loan
   Similarity : 0.6849

4. SBI_Home_Loan_Process_Guide.pdf
   Bank       : SBI
   Loan Type  : Home Loan
   Similarity : 0.6692

5. SBI_Home_Loan_Process_Guide.pdf
   Bank       : SBI
   Loan Type  : Home Loan
   Similarity : 0.6610



The retrieved chunks are organized before being sent to the language model. Duplicate content is removed, higher-ranked chunks are kept first, and the overall context size is controlled to avoid unnecessarily long prompts.

In [31]:
MIN_SIMILARITY = 0.55


def rank_context(retrieved_chunks, max_words=1200):
    """Prepare retrieved chunks for prompt construction."""

    if not retrieved_chunks:
        return []

    # Keep only sufficiently similar chunks
    retrieved_chunks = [
        chunk
        for chunk in retrieved_chunks
        if chunk["similarity_score"] >= MIN_SIMILARITY
    ]

    if not retrieved_chunks:
        return []

    # Highest similarity first
    ranked_chunks = sorted(
        retrieved_chunks,
        key=lambda x: x["similarity_score"],
        reverse=True
    )

    selected_chunks = []
    seen_chunks = set()
    total_words = 0

    for chunk in ranked_chunks:

        text = chunk["chunk_text"].strip()

        if text in seen_chunks:
            continue

        words = chunk["word_count"]

        if total_words + words > max_words:
            break

        selected_chunks.append(chunk)
        seen_chunks.add(text)
        total_words += words

    return selected_chunks

The selected document chunks are merged into a single context block. Only the document content is included here, while source information is preserved separately for the final response.

In [32]:
def build_context(ranked_chunks):
    """Combine retrieved chunks into a single context."""

    if not ranked_chunks:
        return ""

    context = "\n\n".join(
        chunk["chunk_text"]
        for chunk in ranked_chunks
    )

    return context

The chatbot should always mention where its answer comes from. This helper prepares a clean list of source documents without repeating the same file multiple times.

In [33]:
def extract_sources(ranked_chunks):
    """Extract source information."""

    sources = []

    seen = set()

    for chunk in ranked_chunks:

        key = (
            chunk["source_file"],
            chunk["bank"],
            chunk["loan_type"]
        )

        if key in seen:
            continue

        seen.add(key)

        sources.append({
            "source_file": chunk["source_file"],
            "bank": chunk["bank"],
            "loan_type": chunk["loan_type"],
            "similarity_score": round(
                chunk["similarity_score"], 4
            )
        })

    return sources

The response generation step uses the Gemini API. Keeping the API configuration separate makes it easier to switch models or update credentials without changing the rest of the pipeline.

In [34]:
# Install the Gemini SDK (run once)

!pip -q install -U google-generativeai

In [35]:
import google.generativeai as genai
from google.colab import userdata

# Load Gemini API key from Colab Secrets
GEMINI_API_KEY = userdata.get("GEMINI_API_KEY")

genai.configure(api_key=GEMINI_API_KEY)

print("Gemini API configured successfully.")

Gemini API configured successfully.


In [36]:
MODEL_NAME = "gemini-2.5-flash"

llm = genai.GenerativeModel(MODEL_NAME)

print(f"Loaded model: {MODEL_NAME}")

Loaded model: gemini-2.5-flash


The retrieved document chunks are combined with the user's query before sending the request to Gemini. The prompt explicitly instructs the model to answer only from the provided context and avoid unsupported information.

In [37]:
def build_prompt(query, context):
    """Create a grounded prompt for Gemini."""

    prompt = f"""
You are an AI Loan Advisory Assistant.

You must answer ONLY from the context provided below.

Guidelines:
- Never use outside knowledge.
- Never guess or assume information.
- If the context does not contain the answer, reply:
  "The official banking documents do not contain enough information to answer this question."
- If multiple documents provide different values (for example, interest rates), clearly mention that the values depend on the loan scheme or applicant profile.
- Keep the answer professional and easy to understand.
- Use bullet points whenever appropriate.

Context:
{context}

User Question:
{query}

Answer:
"""

    return prompt.strip()

The prepared prompt is sent to Gemini for response generation. Since the model receives only the retrieved document context, the generated answer remains grounded in the official banking documents.

In [38]:
def generate_response(query, ranked_chunks):
    """Generate a response using Gemini."""

    if not ranked_chunks:
        return (
            "The official banking documents do not contain "
            "enough information to answer your question."
        )

    context = build_context(ranked_chunks)
    prompt = build_prompt(query, context)

    try:
        response = llm.generate_content(prompt)

        if hasattr(response, "text") and response.text:
            return response.text.strip()

        return (
            "The official banking documents do not contain "
            "enough information to answer your question."
        )

    except Exception as e:
        return f"Error generating response: {e}"

Along with the generated answer, the chatbot returns the documents used during retrieval. This makes the response transparent and allows users to trace the information back to the original banking documents.

In [39]:
def format_sources(sources):
    """Format retrieved sources."""

    if not sources:
        return "No source documents available."

    formatted = []

    for i, source in enumerate(sources, start=1):

        formatted.append(
            f"{i}. {source['source_file']}\n"
            f"   Bank       : {source['bank']}\n"
            f"   Loan Type  : {source['loan_type']}\n"
            f"   Similarity : {source['similarity_score']:.4f}"
        )

    return "\n\n".join(formatted)

All components are now connected into a single function. Given a user query, the pipeline predicts the intent, retrieves relevant document chunks, generates a grounded response, and returns the supporting source information.

In [40]:
def answer_query(query, top_k=10):
    """Complete RAG pipeline."""

    # Predict intent
    intent_result = predict_intent(query)

    # Apply intent filtering only if prediction is reliable
    if intent_result["confidence"] >= 0.60:

        retrieved_chunks = retrieve_documents(
            query=query,
            intent=intent_result["intent"],
            top_k=top_k
        )

    else:

        retrieved_chunks = retrieve_documents(
            query=query,
            intent=None,
            top_k=top_k
        )

    # Rank retrieved chunks
    ranked_chunks = rank_context(retrieved_chunks)

    # Generate answer
    answer = generate_response(
        query,
        ranked_chunks
    )

    # Extract sources
    sources = extract_sources(ranked_chunks)

    return {
        "query": query,
        "intent": intent_result["intent"],
        "confidence": round(intent_result["confidence"], 4),
        "answer": answer,
        "sources": sources
    }

A few sample banking questions are used to verify that the complete pipeline works correctly. The output includes the predicted intent, generated response, and the documents used during retrieval.

In [41]:
sample_queries = [
    "What is the interest rate for SBI Home Loan?",
    "Which documents are required for an HDFC Car Loan?",
    "How long does SBI Personal Loan approval take?",
    "Can I prepay my education loan without penalty?"
]

for query in sample_queries:

    result = answer_query(query)

    print("=" * 80)
    print(f"Query      : {result['query']}")
    print(f"Intent     : {result['intent']}")
    print(f"Confidence : {result['confidence']:.2f}")

    print("\nAnswer:\n")
    print(result["answer"])

    print("\nSources:\n")
    print(format_sources(result["sources"]))

    print("\n")

Query      : What is the interest rate for SBI Home Loan?
Intent     : interest_rate
Confidence : 0.99

Answer:

The official banking documents do not contain enough information to answer this question. The interest rate is fixed over the tenor of the loan as per the agreement/sanction terms, and customers are directed to `https://bank.sbi/portal/web/interest-rates/other-schemes` for the latest interest rates offered. Interest is calculated at the prevailing rate per annum on a daily reducing balance with monthly rests.

Sources:

1. SBI_PERSONAL_LOAN_MITC.pdf
   Bank       : SBI
   Loan Type  : Personal Loan
   Similarity : 0.6940

2. SBI_Home_Loan_Process_Guide.pdf
   Bank       : SBI
   Loan Type  : Home Loan
   Similarity : 0.6853

3. SBI_Home_Loan_Eligibility.pdf
   Bank       : SBI
   Loan Type  : Home Loan
   Similarity : 0.6606

4. SBI_Home_Loan_Documents_Required.pdf
   Bank       : SBI
   Loan Type  : Home Loan
   Similarity : 0.6542

5. SBI_Personal_Loan_FAQs.pdf
   Bank    

The chatbot should avoid generating unsupported information. These examples help verify that responses remain grounded even when the requested information is unavailable in the official banking documents.

In [42]:
unknown_queries = [
    "Which bank has the best credit card?",
    "Who is the CEO of SBI?",
    "Will RBI reduce interest rates next month?",
    "Suggest the best mutual fund to invest in."
]

for query in unknown_queries:

    result = answer_query(query)

    print("=" * 80)
    print(f"Query : {query}\n")

    print("Answer:\n")
    print(result["answer"])

    print("\nSources:\n")
    print(format_sources(result["sources"]))

    print("\n")

Query : Which bank has the best credit card?

Answer:

The official banking documents do not contain enough information to answer your question.

Sources:

No source documents available.


Query : Who is the CEO of SBI?

Answer:

The official banking documents do not contain enough information to answer your question.

Sources:

No source documents available.


Query : Will RBI reduce interest rates next month?

Answer:

The official banking documents do not contain enough information to answer your question.

Sources:

No source documents available.


Query : Suggest the best mutual fund to invest in.

Answer:

The official banking documents do not contain enough information to answer your question.

Sources:

No source documents available.




In [43]:
query = input("Ask a loan-related question: ")

result = answer_query(query)

print("=" * 80)
print(f"Intent     : {result['intent']}")
print(f"Confidence : {result['confidence']:.2f}")

print("\nAnswer:\n")
print(result["answer"])

print("\nSources:\n")
print(format_sources(result["sources"]))

Ask a loan-related question: What is the interest rate for car loan
Intent     : interest_rate
Confidence : 0.99

Answer:

The interest rate for a car loan:
*   Will be charged at the prevailing fixed rate of interest.
*   Will be determined according to the CIC score of the customer at the time of sanction of the loan.
*   According to the Car Loan EMI Calculator, the interest rate can range between 7% PA and 15% PA.

Sources:

1. SBI_AUTO_LOAN_MITC.pdf.pdf
   Bank       : SBI
   Loan Type  : Vehicle Loan
   Similarity : 0.5817

2. HDFC_Car_Loan_Overview.pdf
   Bank       : HDFC
   Loan Type  : Car Loan
   Similarity : 0.5628


Before integrating the pipeline into the Streamlit application, a few validation checks are performed to ensure that all major components have been loaded correctly and are ready for inference.

In [44]:
print("=" * 70)
print("AI Loan Advisory Chatbot - Pipeline Validation")
print("=" * 70)

print(f"Embedding Model      : {'Loaded' if embedding_model else 'Not Loaded'}")
print(f"FAISS Index          : {faiss_index.ntotal} vectors")
print(f"Metadata Records     : {len(retrieval_metadata)}")
print(f"Intent Classes       : {len(label_encoder.classes_)}")
print(f"LLM                  : {MODEL_NAME}")
print(f"Runtime Device       : {DEVICE}")

print("\nPipeline Status : Ready")

AI Loan Advisory Chatbot - Pipeline Validation
Embedding Model      : Loaded
FAISS Index          : 241 vectors
Metadata Records     : 241
Intent Classes       : 19
LLM                  : gemini-2.5-flash
Runtime Device       : cpu

Pipeline Status : Ready


This notebook completes the Retrieval-Augmented Generation pipeline for the AI Loan Advisory Chatbot. It combines intent prediction, semantic retrieval, context ranking and Gemini-based response generation into a reusable workflow. The same functions developed here will be used directly in the Streamlit application without modifying the backend logic.

In [45]:
for chunk in retrieval_metadata:

    if (
        "interest" in chunk["chunk_text"].lower()
        and chunk["loan_type"] == "Home Loan"
        and chunk["bank"] == "SBI"
    ):

        print("=" * 80)
        print(chunk["source_file"])
        print(chunk["chunk_text"])

SBI_Home_Loan_Agreement.pdf
8. SBI Green Home Loan
Purpose for which home loan can be availed: The loan will be sanctioned for the purpose of purchase
/ construction / extension / repairs/renovation of new/second-hand residential house/flat/plot of land and
furnishings (hereinafter referred to as the ‘project’).
Premium of Home Loan Insurance cover (Optional) : The premium for the optional Home Loan Life
Insurance cover (if availed) will be added to the loan amount.
Loan to Value Ratio (LTV):
For loan amount upto Rs.20 Lacs, maximum permissible LTV ratio is 90% of the assessed value of the
property. For loan amount greater than Rs.20 Lacs and upto Rs.75 Lacs, maximum permissible LTV ratio
is 80%. A maximum permissible LTV ratio of 75% is applicable on a loan amount above Rs.75 Lacs.
of Interest:
Floating of Interest: -
Interest on the loan will be charged at prevailing floating rate of interest on a daily reducing balance at
monthly rests. The rate of interest is subject to revision fr